In [3]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path

import yaml

# LangChain의 핵심 컴포넌트들을 가져옵니다.
from langchain_core.prompts import PromptTemplate

PROMPTS_PATH = (
    Path(os.getcwd()).parent
    / "chatbot"
    / "langgraph_core"
    / "prompts"
    / "system_prompts.yaml"
)


# --- 1. 신뢰할 수 있는 소스(YAML)에서 템플릿 로드 ---
# 이 함수는 우리가 직접 통제하는 YAML 파일에서 프롬프트 문자열을 안전하게 로드하는 역할을 합니다.
def load_template_from_trusted_source(file_path: str, key: str) -> str:
    """
    지정된 YAML 파일에서 특정 키에 해당하는 프롬프트 템플릿 문자열을 로드합니다.
    이 함수는 '신뢰할 수 있는 출처'에서만 템플릿을 가져오는 것을 보장합니다.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Prompt file not found at path: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        # yaml.safe_load를 사용하여 안전하게 파싱합니다.
        all_prompts = yaml.safe_load(f)

    # 중첩된 키를 점(.)으로 접근하여 값을 가져옵니다 (예: "onboarding.system").
    keys = key.split(".")
    template = all_prompts
    for k in keys:
        template = template[k]
    return template


# --- 2. LangChain PromptTemplate 생성 및 사용 ---
if __name__ == "__main__":
    try:
        # --- 단계 1: 신뢰할 수 있는 YAML 파일에서 Jinja2 템플릿 문자열 로드 ---
        # 이 과정은 해커가 개입할 수 없는, 개발자가 완전히 통제하는 과정입니다.
        trusted_template_string = load_template_from_trusted_source(
            file_path=PROMPTS_PATH, key="onboarding.system"
        )
        print("--- YAML에서 로드한 원본 Jinja2 템플릿 ---")
        print(trusted_template_string)
        print("\n" + "=" * 50 + "\n")

        # --- 단계 2: PromptTemplate.from_template 클래스 메서드 사용 ---
        # 로드한 '신뢰할 수 있는' 템플릿 문자열을 사용하여 PromptTemplate 객체를 생성합니다.
        # template_format을 'jinja2'로 명시하여 Jinja2 템플릿 엔진을 사용하도록 지시합니다.
        # 이 시점에서 LangChain은 내부적으로 Jinja2의 SandboxedEnvironment를 사용합니다.
        prompt = PromptTemplate.from_template(
            template=trusted_template_string,
            template_format="jinja2",
        )

        # --- 단계 3: 시나리오별로 프롬프트 포맷팅 (변수 주입) ---

        # 시나리오 A: 신규 사용자
        # Jinja2 템플릿의 `{% if not user_data ... %}` 조건에 따라 분기됩니다.
        print("--- 시나리오 A: 신규 사용자 ---")
        # .format() 메서드에 템플릿에서 사용하는 변수명(user_data)을 키로 전달합니다.
        formatted_prompt_new_user = prompt.format(user_data=None)
        print(formatted_prompt_new_user)
        print("\n" + "-" * 50 + "\n")

        # 시나리오 B: 기존 사용자
        # Jinja2 템플릿의 `{% else %}` 조건에 따라 분기됩니다.
        returning_user_profile = {
            "name": "김민준",
            "investment_goal": "주택 자금 마련",
            "interests": ["AI", "ESG", "반도체"],
        }
        print("--- 시나리오 B: 기존 사용자 ---")
        formatted_prompt_returning_user = prompt.format(
            user_data=returning_user_profile
        )
        print(formatted_prompt_returning_user)

    except FileNotFoundError as e:
        print(f"오류: {e}")
    except KeyError as e:
        print(f"오류: YAML 파일에서 키를 찾을 수 없습니다 - {e}")
    except Exception as e:
        print(f"알 수 없는 오류가 발생했습니다: {e}")

--- YAML에서 로드한 원본 Jinja2 템플릿 ---
You are a friendly and empathetic AI investment advisor chatbot. Your primary goal is to build rapport with the user and provide personalized investment guidance.

**Your Persona & Language:**
- Your name is "자산구조원 AI"
- This program's name is "FIREgenerator"
- You MUST communicate in natural, empathetic, and polite **Korean**.
- Your tone should be patient, understanding, and never judgmental, encouraging. Use phrases like:
  - "만나서 반갑습니다! 앞으로 잘 부탁드려요."
  - "그렇게 생각하실 수 있군요. 충분히 이해됩니다."
  - "좋은 목표를 가지고 계시네요! 함께 차근차근 알아가 봐요."
- Avoid financial jargon. If you must use it, explain it in simple terms.

**Conversation Flow:**
{% if not user_data or not user_data.name %}
# --- New User Flow ---
Your primary mission is to have a natural conversation to gather the following information.
Do NOT ask for all information at once. Weave your questions into the conversation organically, one by one.

1.  **이름 (Name):** Start by asking for th을ir name to personalize the